# CircularNet - Waste identification with instance segmentation
Welcome to the Instance Segmentation Notebook! This notebook will take you through the steps of running an  Instance Segmentation model on Images. \
There are two ways to run it :
1. Pytorch Model : How to quickly run and test the CircularNet model's Pytorch version.
2. ONNX Model : Running the converted ONNX version of the model.

# 1. Pytorch Model

## Install Required python packages

In [ ]:
!pip install -q rfdetr==1.5.2 supervision

## Import Libraries

In [ ]:
import json
import os
import glob
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import cv2
from PIL import Image
import supervision as sv
from rfdetr import RFDETRSegMedium

## Load Image Paths

In [ ]:
LIMIT = 20
IMAGE_DIR_PATH = "./images"

all_images = glob.glob(f"{IMAGE_DIR_PATH}/*")[:LIMIT]
print(f"Number of Images : {len(all_images)})

## Load Segmentation Model

In [ ]:
!wget https://storage.googleapis.com/tf_model_garden/vision/waste_identification_ml/CN-ModelCheckpoints/ModelRegistry_432x432_March26/checkpoint_best_total.pth

In [ ]:
seg_medium_model = RFDETRSegMedium(pretrain_weights="./checkpoint_best_total.pth")
class_id_to_class_name_mapper = seg_medium_model.class_names

## Run the model

In [ ]:
detections = [seg_medium_model.predict(img_path, threshold=0.5) for img_path in tqdm(all_images)]

## Visualize Results

In [ ]:
detections_images = []
for i in np.arange(len(all_images)-1):
    path = all_images[i]
    image = Image.open(path)

    detection = detections[i]

    text_scale = sv.calculate_optimal_text_scale(resolution_wh=image.size)
    thickness = sv.calculate_optimal_line_thickness(resolution_wh=image.size)
    color = sv.ColorPalette.from_hex([
        "#ffff00", "#ff9b00", "#ff66ff", "#3399ff", "#ff66b2", "#ff8080",
        "#b266ff", "#9999ff", "#66ffff", "#33ff99", "#66ff66", "#99ff00"
    ])

    bbox_annotator = sv.BoxAnnotator(color=color, thickness=thickness)
    mask_annotator = sv.MaskAnnotator()

    label_annotator = sv.LabelAnnotator(
        color=color,
        text_color=sv.Color.BLACK,
        text_scale=text_scale)

    detections_labels = [
        f"{class_id_to_class_name_mapper[int(class_id+1)]} {confidence:.2f}"
        for class_id, confidence
        in zip(detection.class_id, detection.confidence)
    ]

    detections_image = image.copy()
    detections_image = mask_annotator.annotate(detections_image, detection)
    detections_image = bbox_annotator.annotate(detections_image, detection)
    detections_image = label_annotator.annotate(detections_image, detection, detections_labels)

    detections_images.append(detections_image)

# adjust the grid size based upon number of images, below it is set to 5(rows) x 4 (columns) for viewing 20 images
sv.plot_images_grid(images=detections_images, grid_size=(5, 4), size=(20, 20))

# 2. How to run ONNX Version of the Model

## Download model and essential codebase

In [ ]:
!git clone --depth 1 https://github.com/tensorflow/models.git
!wget https://storage.googleapis.com/tf_model_garden/vision/waste_identification_ml/CN-ModelCheckpoints/ModelRegistry_432x432_March26/inference_model.onnx
!wget https://storage.googleapis.com/tf_model_garden/vision/waste_identification_ml/CN-ModelCheckpoints/sample_image.jpg
!mv models/official/projects/waste_identification_ml/Deploy/detr_cloud_deployment/client/labels50.csv ./

## Install all the required libraries


In [ ]:
!pip install -q onnx==1.19.1 onnxruntime==1.23.2 supervision tritonclient[http]==2.58.0

## Import Python Libraries

In [ ]:
import onnx
import onnxruntime as ort

from PIL import Image
import supervision as sv

import sys
from unittest.mock import MagicMock
sys.modules["color_extraction"] = MagicMock()

from models.official.projects.waste_identification_ml.Deploy.detr_cloud_deployment.client.triton_server_inference import TritonObjectDetector
from models.official.projects.waste_identification_ml.Deploy.detr_cloud_deployment.client.utils import draw_detections_and_save_image

## Define essential variables

In [ ]:
input_image_path = "./sample_image.jpg"
onnx_model_path = "./inference_model.onnx"
output_image_dimension = (1920, 1080)

## Initialize ONNX version of the model

In [ ]:
model_utils = TritonObjectDetector()
onnx_model = ort.InferenceSession(onnx_model_path)

## Do model Inferencing

In [ ]:
image_array = model_utils._get_input_batch_for_inference(image_path=input_image_path)
outputs = onnx_model.run(None, {"input": image_array})

## Format the model output as per output dimensions

In [ ]:
results = model_utils._reformat_triton_output_to_dict(
    outputs, confidence_threshold=0.5, max_boxes=100
)

# output_image_dimension makes sure the output image dimession along with the bounding boxes and masks are resized.
results = model_utils._scale_bbox_and_masks(results, target_dims=output_image_dimension)
results["class_names"] = model_utils.get_class_names(results)

## Save output and Visualize Results

In [ ]:
draw_detections_and_save_image(img=Image.open(input_image_path), results=results, save_path= "./output.jpg")
Image.open("./output.jpg")

# END of Notebook